# News 기반 QA RAG 구현

## Navie RAG

따로 문서를 저장한게 아니라 웹 베이스 로더를 통해서 문서 로드하고 나누고 벡터 스토어 만들고... 이 9단계 그대로 거친다. 프롬프트 기반으로 체이닝해서 질문 답변 똑같이 해준다!

In [ ]:
import bs4 # 뉴스 기반이면 이걸 넣어야겠지
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# 뉴스기사 내용을 로드하고, 청크로 나누고, 인덱싱합니다.
loader = WebBaseLoader(
    web_paths=("https://n.news.naver.com/mnews/article/009/0005680884",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class": ["newsct_article_article_body", "media_end_head_title"]}, # div 태그 중에서도 class가 둘 중 하나인 div만 가져온다. 
        )
    ),
) # 웹 뉴스 페이지에서 필요한 html 영역만 골라서 문서로 로드하는 코드임 # HTML 전체 다 읽지 않고 div만 대상으로 본다!
# 태그와 클래스로 미루어 봤을 때 네이버 뉴스에서 제목과 본문만 doc로 가져올 것을 예상할 수 있다. 

docs = loader.load()
print(f"문서의 수: {len(docs)}")
docs

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
splits = text_splitter.split_documents(docs)

vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()
print(f"FAISS 인덱스 생성 완료: documents={len(splits)}")


In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """당신은 질문-답변(Question-Answering)을 수행하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context) 에서 주어진 질문(question) 에 답하는 것입니다.
검색된 다음 문맥(context) 을 사용하여 질문(question) 에 답하세요. 
만약, 주어진 문맥(context) 에서 답을 찾을 수 없다면, 답을 모른다면 `주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다` 라고 답하세요.
한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요.
그리고 답변시에 필요한 정보를 바탕으로 단계별로 생각해서 답해줘.

## INSTRUCTION


## ANSWER FORMAT


#Question: 
{question} 

#Context: 
{context} 

#Answer:
"""
)


llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)


# 체인을 생성합니다.
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:
answer = rag_chain.invoke("앤트로픽의 애플 해킹에 대해서 설명해줘")
print(answer)

# 예상 답변: 1.  **앤트로픽의 AI 해커 개발:** 주어진 문맥에 따르면, 앤트로픽은 "최악의 AI해커"를 만들었습니다.
2.  **애플 시스템 해킹:** 이 앤트로픽이 만든 AI 해커로 인해, "애플은 안 뚫립니다"라는 인식이 있었음에도 불구하고 5일 뒤 애플 시스템이 "뚫렸다"고 언급됩니다.

**이처럼 web상의 자료를 로드해서 우리의 DB에 분할 저장하고, 이를 기반으로 답변을 생성하는 RAG기반 llm을 만들 수 있다!!**